# Train a PLAUD model in Colab

**PLAUD** turns a folder of your own audio into a realtime generative instrument for
[`nn~`](https://github.com/acids-ircam/nn_tilde) (Max/MSP & PureData).

This notebook runs the entire training process in thre stages:

1. **Synth** — a DDSP, NoiseBandNet-based model learns to resynthesize your sound.
2. **Codec + Prior** — your sound is translated to latent trajectories, which in are in turn compressed into tokens, and a Transformer learns to *generate*
   new sequences of them (this is the part that improvises in your style).
3. **Export** — everything is packed into a single `models/<name>.ts` file you load in `nn~`.

All you have to do is **point it at a folder of audio** and press ▶ on each cell, top to bottom.

> ⚠️ **A GPU is required.** Go to **Runtime → Change runtime type → Hardware accelerator → GPU (T4)**
> *before* you start. Training on CPU will not work.


## 1 · Setup

Checks that a GPU is available, downloads PLAUD, and installs everything it needs. Takes a couple of
minutes. Run it once per session.


In [ ]:
#@title Run setup { display-mode: "form" }
import torch
assert torch.cuda.is_available(), \
    "No GPU! Go to Runtime → Change runtime type → GPU (T4), then re-run this cell."
print("✅ GPU:", torch.cuda.get_device_name(0))

import os
if not os.path.isdir('/content/plaud'):
    !git clone -b postnet https://github.com/blazejkotowski/plaud.git /content/plaud
%cd /content/plaud

# Optional submodules (not needed for the default synth) — rewrite SSH→HTTPS, ignore failures.
!git -c url."https://github.com/".insteadOf="git@github.com:" submodule update --init --recursive || true

# System tool used to convert/resample audio.
!apt-get -qq install -y sox
# Python deps: the repo's requirements + a few the code imports but requirements.txt omits.
!pip install -q -r requirements.txt
!pip install -q hydra-core omegaconf soundfile tqdm scikit-learn cached_conv
!pip install -q -e .

print("\n✅ Setup complete.")

## 2 · Get your audio in

You can train on audio from **Google Drive** *or* from files you **upload** into this Colab session.

- **Google Drive** — run the cell below to mount it, then point the next cell at a folder inside
  `\/content\/drive\/MyDrive\/...`.
- **Upload** — open the 📁 **Files** panel on the left, drag a folder of audio into `\/content`, and
  point the next cell at it (e.g. `\/content\/my_sounds`). *(Uploaded files are lost when the session
  ends; Drive is safer for anything you want to keep.)*

Any common format works: `wav`, `mp3`, `flac`, `ogg`, `aiff`, … A single file is enough to start;
more varied material simply gives the **Style pad** in `nn~` more to explore.

In [ ]:
#@title (Optional) Mount Google Drive { display-mode: "form" }
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title Point to your audio folder { display-mode: "form" }
#@markdown Path to the folder that holds your audio files:
AUDIO_DIR = ""  #@param {type:"string"}

import os, glob
assert AUDIO_DIR and os.path.isdir(AUDIO_DIR), f"Folder not found: {AUDIO_DIR!r}"
_EXTS = ('.wav', '.mp3', '.flac', '.ogg', '.aac', '.aiff')
_files = [f for f in glob.glob(os.path.join(AUDIO_DIR, '**', '*'), recursive=True)
          if f.lower().endswith(_EXTS)]
assert len(_files) >= 1, "No audio files found in that folder — check the path."
print(f"✅ Found {len(_files)} audio file(s).")

## 3 · Project settings

The only choices you need to make. Sensible defaults are filled in — a first run is fine as-is.


In [ ]:
#@title Project settings { display-mode: "form" }
MODEL_NAME = "my_model"      #@param {type:"string"}
SAMPLE_RATE = "44100"        #@param ["44100", "48000"]
CHANNELS = "mono"            #@param ["mono", "stereo"]
NORMALIZE_LOUDNESS = True    #@param {type:"boolean"}

SAMPLE_RATE = int(SAMPLE_RATE)
N_CHANNELS = 1 if CHANNELS == "mono" else 2
MODEL_NAME = MODEL_NAME.strip().replace(" ", "_") or "my_model"
print(f"model = {MODEL_NAME}   sample_rate = {SAMPLE_RATE} Hz   channels = {CHANNELS}   "
      f"loudness-normalize = {NORMALIZE_LOUDNESS}")

## 4 · Prepare the dataset

Converts every file to the sample rate / channel count above and (optionally) loudness-matches them
so no single track dominates. The result is written to a fast local folder that training reads from.


In [ ]:
#@title Convert & resample { display-mode: "form" }
DATASET_PATH = f"/content/plaud_data/{MODEL_NAME}"

cmd = (f'python -m utils.dataset_converter '
       f'--input_dir "{AUDIO_DIR}" --output_dir "{DATASET_PATH}" --sampling_rate {SAMPLE_RATE}')
if N_CHANNELS == 1:
    cmd += ' --channels 1'
if NORMALIZE_LOUDNESS:
    cmd += ' --normalize_rms 0.1'
print(cmd)
!{cmd}

import glob
_n = len(glob.glob(os.path.join(DATASET_PATH, '*.wav')))
assert _n >= 1, "Conversion produced no files — is `sox` installed and the input readable?"
print(f"\n✅ Prepared {_n} wav file(s) → {DATASET_PATH}")

## 5 · How long to train

The pipeline has three training stages. The defaults below are a **quick first pass** (≈ 1–2 h total
on a free T4) — enough to hear whether it's working. For a **final, high-quality** model use the
recommended values noted on each slider; that can run for many hours.

Every stage **saves checkpoints and resumes automatically**, so you can start with the quick
defaults, listen, then raise these numbers and re-run a stage to keep training where it left off.


In [ ]:
#@title Training length { display-mode: "form" }
SYNTH_EPOCHS = 30        #@param {type:"integer"}
#@markdown &nbsp;&nbsp;↑ **Synth** epochs — quick: 30 · recommended: ~100
COMPRESSOR_EPOCHS = 60   #@param {type:"integer"}
#@markdown &nbsp;&nbsp;↑ **Codec** epochs — quick: 60 · recommended: ~250
PRIOR_STEPS = 15000      #@param {type:"integer"}
#@markdown &nbsp;&nbsp;↑ **Prior** steps — quick: 15000 · recommended: ~50000 (the sweet spot)

print(f"synth = {SYNTH_EPOCHS} epochs · codec = {COMPRESSOR_EPOCHS} epochs · prior = {PRIOR_STEPS} steps")

## 6 · Build the config

Writes a `configs/<name>.yaml` from the repo's ready-to-train recipe (`configs/template.yaml` — the
discrete codec + learned style + LFO prior), filling in your settings above. All three stages read
this one file. The synth batch size is chosen to fit a free T4's GPU memory.

In [ ]:
#@title Write the config { display-mode: "form" }
import yaml

BASE_CFG = "configs/template.yaml"   # the annotated, ready-to-train discrete codec + style + LFO recipe
with open(BASE_CFG) as f:
    cfg = yaml.safe_load(f)

cfg["experiment"]["name"]        = MODEL_NAME
cfg["data"]["dataset_path"]      = DATASET_PATH
cfg["audio"]["fs"]               = SAMPLE_RATE
cfg["audio"]["n_channels"]       = N_CHANNELS
cfg["trainer"]["max_epochs"]     = int(SYNTH_EPOCHS)
cfg["compressor"]["max_epochs"]  = int(COMPRESSOR_EPOCHS)
cfg["prior"]["training"]["max_steps"] = int(PRIOR_STEPS)

# Keep the synth stage within a free T4's ~15 GB of VRAM: batch 6 is fine for mono/44.1 kHz;
# stereo or 48 kHz roughly doubles the memory, so drop to 4 there.
cfg["trainer"]["batch_size"] = 6 if (N_CHANNELS == 1 and SAMPLE_RATE <= 44100) else 4

CONFIG_NAME = MODEL_NAME
out_path = f"configs/{CONFIG_NAME}.yaml"
with open(out_path, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print(f"✅ wrote {out_path}  (synth batch_size = {cfg['trainer']['batch_size']})\n")
print(open(out_path).read())

## 7 · Stage 1 — train the synth

Teaches the DDSP model to reproduce your sound. This is usually the longest stage. You can re-run
this cell to resume from the last checkpoint.


In [ ]:
#@title Train the synth { display-mode: "form" }
!python -m cli.train -cn {CONFIG_NAME} hydra.job.chdir=False

## 8 · Stage 2 — train the codec + prior

First compresses the synth's control signals into tokens (the **codec**), then trains the
**prior** — the Transformer that generates new token sequences in your style. Both run from this one
cell.


In [ ]:
#@title Train the codec + prior { display-mode: "form" }
!python -m cli.train_prior -cn {CONFIG_NAME} hydra.job.chdir=False

## 9 · Export to `nn~`

Packs the synth, codec, and prior into a single TorchScript file, `models/<name>.ts` — the file you
load in `nn~`. It also renders a **Style-pad terrain map** (`_terrain.json` + `.png`) that the Max
patch can display. The terrain render adds a few minutes.

In [ ]:
#@title Export the model { display-mode: "form" }
import os
os.makedirs("models", exist_ok=True)
# Also emits a Style-pad "terrain" map (JSON + PNG) for the nn~ patch. terrain_grid=12 keeps the
# render to a few minutes on a T4 (the default 24 is much slower); raise it for a finer map.
!python -m cli.export --config {CONFIG_NAME} --type last --target_fs {SAMPLE_RATE} --terrain_grid 12

TS_PATH = f"models/{MODEL_NAME}.ts"
assert os.path.isfile(TS_PATH), "Export failed — check the log above."
print(f"\n✅ {TS_PATH}   ({os.path.getsize(TS_PATH) / 1e6:.1f} MB)")
for ext in ("_terrain.json", "_terrain.png"):
    p = f"models/{MODEL_NAME}{ext}"
    if os.path.isfile(p):
        print(f"   terrain: {p}")

## 10 · Save your model

If Google Drive is mounted, the model is copied there; otherwise it downloads to your computer.


In [ ]:
#@title Save / download { display-mode: "form" }
import os, shutil
TS_PATH = f"models/{MODEL_NAME}.ts"
if os.path.isdir("/content/drive/MyDrive"):
    dst_dir = "/content/drive/MyDrive/plaud_models"
    os.makedirs(dst_dir, exist_ok=True)
    dst = os.path.join(dst_dir, f"{MODEL_NAME}.ts")
    shutil.copy(TS_PATH, dst)
    print(f"✅ copied to {dst}")
else:
    from google.colab import files
    files.download(TS_PATH)

## 11 · Play it in `nn~`

Install the [`nn~`](https://github.com/acids-ircam/nn_tilde) external for Max/MSP or PureData and
load your `models/<name>.ts`. The instrument's `prior` generates control, and `decode` renders it to
audio. The headline live controls are:

- **Style X / Y** — an XY pad that morphs between the styles in your dataset.
- **Temperature** — how varied the generation is (start around **0.6**).
- **Style CFG** — how strongly it commits to the pad's style (start around **1.5**).

There are more timbre controls (waveshaping, spectral bends, partial limiting) exposed as `nn~`
attributes. See the repo's **README → "Realtime control surface"** and **`DISCRETE_PRIOR.md`** for
the full tuning guide.
